In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

#read data
train_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')
train_df.info()

In [ ]:
#Histogram of SalesPrice
plt.figure(figsize=(10,6))
sns.histplot(train_df['SalePrice'], kde=True)
plt.title('Distribution of SalePrice')
plt.show()

In [ ]:
#Apply a log transformation to SalePrice
#We use np.log1p() which calculates log(1+x) to handle potential zero values gracefully
train_df['SalePrice_log'] = np.log1p(train_df['SalePrice'])

#Create a figure with two subplots to compare
fig, ax = plt.subplots(1,2, figsize=(16,6))

#Origin distribution
sns.histplot(train_df['SalePrice'], kde=True, ax=ax[0])
ax[0].set_title('Original Distribution of SalePrice')

#Log-transformed distribution
sns.histplot(train_df['SalePrice_log'], kde=True, ax=ax[1])
ax[1].set_title('Log-transformeddistribution')

plt.show()

In [ ]:
#Calculate the percentage of missing values in each column
missing_ratio = train_df.isnull().sum() / len(train_df)
missing_ratio = missing_ratio[missing_ratio > 0].sort_values(ascending=False)
print(missing_ratio)

In [ ]:
#Drop columns with too many missing values
train_df = train_df.drop(columns=['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu'])
test_df = test_df.drop(columns=['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu'])

#Fill missing values in categorical valiables with 'None'
for col in ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']:
    train_df[col] = train_df[col].fillna('None')
    test_df[col] = test_df[col].fillna('None')

#Fill missing values in numerical variables with the median
lot_frontage_median = train_df['LotFrontage'].median()
train_df['LotFrontage'] = train_df['LotFrontage'].fillna(lot_frontage_median)
test_df['LotFrontage'] = test_df['LotFrontage'].fillna(lot_frontage_median)

# Check the number of missing values after processing
print("\nNumber of missing values in training data after processing:")
print(train_df.isnull().sum().sum())

In [ ]:
# Fill missing values in basement-related categorical variables with 'None'
for col in ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']:
    train_df[col] = train_df[col].fillna('None')
    test_df[col] = test_df[col].fillna('None')

#Handle other potentially remaining missing values
electrical_mode = train_df['Electrical'].mode()[0]
train_df['Electrical'] = train_df['Electrical'].fillna(electrical_mode)

# Fill missing values that exist only in the test data using the median or mode of each column
test_df['MSZoning'] = test_df['MSZoning'].fillna(test_df['MSZoning'].mode()[0])
test_df['Utilities'] = test_df['Utilities'].fillna(test_df['Utilities'].mode()[0])
test_df['Exterior1st'] = test_df['Exterior1st'].fillna(test_df['Exterior1st'].mode()[0])
test_df['Exterior2nd'] = test_df['Exterior2nd'].fillna(test_df['Exterior2nd'].mode()[0])
test_df['BsmtFinSF1'] = test_df['BsmtFinSF1'].fillna(test_df['BsmtFinSF1'].median())
test_df['BsmtFinSF2'] = test_df['BsmtFinSF2'].fillna(test_df['BsmtFinSF2'].median())
test_df['BsmtUnfSF'] = test_df['BsmtUnfSF'].fillna(test_df['BsmtUnfSF'].median())
test_df['TotalBsmtSF'] = test_df['TotalBsmtSF'].fillna(test_df['TotalBsmtSF'].median())
test_df['BsmtFullBath'] = test_df['BsmtFullBath'].fillna(test_df['BsmtFullBath'].mode()[0])
test_df['BsmtHalfBath'] = test_df['BsmtHalfBath'].fillna(test_df['BsmtHalfBath'].mode()[0])
test_df['KitchenQual'] = test_df['KitchenQual'].fillna(test_df['KitchenQual'].mode()[0])
test_df['Functional'] = test_df['Functional'].fillna(test_df['Functional'].mode()[0])
test_df['GarageCars'] = test_df['GarageCars'].fillna(test_df['GarageCars'].median())
test_df['GarageArea'] = test_df['GarageArea'].fillna(test_df['GarageArea'].median())
test_df['SaleType'] = test_df['SaleType'].fillna(test_df['SaleType'].mode()[0])

# Most missing values should be handled by now
# Check the count of missing values again
print("Total remaining missing values in training data:", train_df.isnull().sum().sum())
print("Total remaining missing values in test data:", test_df.isnull().sum().sum())

In [ ]:
#Combine training and test data to process them at once
all_df = pd.concat([train_df.drop(['SalePrice_log'], axis=1), test_df],sort=False)

#Apply One-Hot Encoding
all_df = pd.get_dummies(all_df)

all_df_imputed = all_df.fillna(all_df.median())
print("Number of remaining NaNs after this new step:", all_df_imputed.isnull().sum().sum())

# Split the data back into train and test sets
train_processed_df = all_df_imputed[:len(train_df)]
test_processed_df = all_df_imputed[len(train_df):]

print("Number of columns in training data after One-Hot Encoding:", train_processed_df.shape[1])

In [ ]:
# Import the LightGBM library
import lightgbm as lgb

# --- 1. Prepare the data (same as before) ---
# X_train, y_train, and X_test should be ready from the previous steps
X_train = train_processed_df
y_train = train_df['SalePrice_log']
X_test = test_processed_df

# --- 2. Set up and train the LightGBM model ---
# Create a LightGBM Regressor model
# Using some common starting parameters for good performance
model_lgbm = lgb.LGBMRegressor(objective='regression', 
                               num_leaves=31,
                               learning_rate=0.05,
                               n_estimators=1000,
                               random_state=42)

# Train the model on the entire training data
model_lgbm.fit(X_train, y_train)

# --- 3. Make predictions ---
# Predict on the test data
predictions_log = model_lgbm.predict(X_test)

# --- 4. Convert predictions back to original scale ---
# Transform the log-predictions back to dollar amounts
predictions = np.expm1(predictions_log)

# --- 5. Create the submission file ---
submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": predictions
})
submission.to_csv('submission_lightgbm.csv', index=False)

print("submission_lightgbm.csv file has been created successfully!")